# Notebook D: Q&A Dataset Preparation

**Goal**: Load the synthetic Spurgeon Q&A dataset (`spurgeon_train_1500.jsonl`), parse it into a Hugging Face `Dataset` structure, apply the Qwen2.5 ChatML template, split it into train/val sets (95/5), and save it to disk for training.

## 1. Environment Detection & Setup

In [ ]:
import os
from pathlib import Path

# Detect environment
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "COLAB_GPU" in os.environ

if IS_KAGGLE:
    print("Running on Kaggle")
    INPUT_DATASET_PATH = "/kaggle/input/spurgeon-qa-dataset/spurgeon_train.jsonl"
    OUTPUT_DIR = "/kaggle/working/"
elif IS_COLAB:
    print("Running on Google Colab")
    # Expect file to be uploaded or loaded from drive
    INPUT_DATASET_PATH = "/content/spurgeon_train.jsonl"
    OUTPUT_DIR = "/content/"
else:
    print("Running locally")
    INPUT_DATASET_PATH = "../data/spurgeon_train.jsonl"
    OUTPUT_DIR = "../data/"

print(f"Input path: {INPUT_DATASET_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

## 2. Install Dependencies (Kaggle/Colab only)

In [ ]:
if IS_KAGGLE or IS_COLAB:
    !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install -q --no-deps xformers "trl" peft accelerate bitsandbytes datasets

## 3. Load JSONL Dataset

In [ ]:
import json
from datasets import Dataset

records = []
if not Path(INPUT_DATASET_PATH).exists():
    raise FileNotFoundError(f"Dataset not found at {INPUT_DATASET_PATH}. Please upload it first.")

with open(INPUT_DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print(f"Loaded {len(records):,} total training examples.")

# Load as HF Dataset
dataset = Dataset.from_list(records)
print(dataset)

## 4. Tokenize & Format with ChatML Template

In [ ]:
from unsloth import FastLanguageModel

# Load model and tokenizer (we just need tokenizer for templates)
model_name = "unsloth/Qwen2.5-3B-Instruct" if not IS_KAGGLE else "/kaggle/input/spurgeon-phase1-merged"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Apply ChatML template for Qwen2.5 (required since base models lack default templates)
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def format_chatML(example):
    messages = example["messages"]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": formatted}

# Apply the mapping
dataset = dataset.map(format_chatML)
print("Sample formatted example:")
print(dataset[0]["text"][:800])

## 5. Train / Val Split

In [ ]:
# Split 95% train / 5% val
split_dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split_dataset["train"]
val_ds = split_dataset["test"]

print(f"Train examples: {len(train_ds):,}")
print(f"Val examples: {len(val_ds):,}")

## 6. Save Datasets to Disk

In [ ]:
train_path = os.path.join(OUTPUT_DIR, "qa_dataset_train")
val_path = os.path.join(OUTPUT_DIR, "qa_dataset_val")

train_ds.save_to_disk(train_path)
val_ds.save_to_disk(val_path)

print(f"Saved train dataset to: {train_path}")
print(f"Saved val dataset to: {val_path}")